# Intent Classification Model Fine-tuning (5 Classes)

This notebook fine-tunes a transformer-based classification model for intent detection, simplified to **5 core classes** for better accuracy and stability.

**Improvements from v2**:
- Simplified taxonomy (5 broad intents instead of 14 granular ones)
- Higher training stability due to balanced classes
- Better focus on the core booking workflow

**Task**: Multi-class classification (5 intent classes)  
**Model**: BERT-base-uncased  
**Data**: Bilingual (English + Roman Urdu) booking assistant messages

---

## 1. Setup & Dependencies

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset as HFDataset, DatasetDict

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration & Class Mapping

In [ ]:
# Paths
DATA_DIR = Path("../data")
DATASET_PATH = DATA_DIR / "intent_dataset_merged.jsonl"
OUTPUT_DIR = Path("./output_v3") # New output dir
MODEL_SAVE_PATH = OUTPUT_DIR / "intent_classifier_v3"

MODEL_NAME = "bert-base-uncased"

# Training hyperparameters
CONFIG = {
    "max_length": 128,                    # Max token length
    "batch_size": 16,                     # Can be larger for 5 classes
    "gradient_accumulation_steps": 1,     # Effective batch size = 16
    "learning_rate": 2e-5,                # Standard fine-tuning rate
    "num_epochs": 10,                     # Sufficient for this task
    "weight_decay": 0.01,                 # L2 regularization
    "warmup_ratio": 0.1,                  # 10% warmup steps
    "eval_steps": 50,                     # Evaluate every 50 steps
    "save_steps": 50,                     # Save every 50 steps
    "use_focal_loss": True,               # Use Focal Loss for class imbalance
    "focal_gamma": 2.0,                   # Gamma for focal loss
    "augment_data": True,                 # Enable data augmentation
    "min_samples_per_class": 50,          # Higher minimum samples for simplified classes
}

# 5 CORE CLASSES
INTENT_LABELS = [
    "greeting",
    "inquiry",
    "info",
    "transaction_confirm",
    "unknown"
]

# MAPPING FROM 14 CLASSES TO 5 CLASSES
INTENT_MAPPING = {
    "greeting": "greeting",
    "booking_request": "inquiry",
    "availability_inquiry": "inquiry",
    "service_selection": "inquiry",
    "date_selection": "inquiry",
    "time_selection": "inquiry",
    "price_inquiry": "info",
    "information": "info",
    "payment_related": "info",
    "confirmation": "transaction_confirm",
    "cancellation": "transaction_confirm",
    "modification": "transaction_confirm",
    "name_provided": "unknown",
    "unknown": "unknown"
}

# Create label mappings
label2id = {label: idx for idx, label in enumerate(INTENT_LABELS)}
id2label = {idx: label for idx, label in enumerate(INTENT_LABELS)}

print(f"Target Classes: {INTENT_LABELS}")
print(f"Number of classes: {len(INTENT_LABELS)}")
print(f"Model: {MODEL_NAME}")
print(f"Use Focal Loss: {CONFIG['use_focal_loss']}")

## 3. Custom Loss Functions

In [ ]:
class WeightedCrossEntropyLoss(nn.Module):
    """Weighted Cross Entropy Loss for imbalanced classes."""
    def __init__(self, class_weights):
        super().__init__()
        self.class_weights = class_weights
    
    def forward(self, logits, labels):
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        return loss_fn(logits, labels)


class FocalLoss(nn.Module):
    """Focal Loss for imbalanced classification.
    
    Focal Loss addresses class imbalance by down-weighting easy examples
    and focusing on hard examples.
    """
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha  # Class weights (tensor)
        self.gamma = gamma  # Focusing parameter
    
    def forward(self, logits, labels):
        ce_loss = nn.CrossEntropyLoss(reduction='none', weight=self.alpha)(logits, labels)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

print("Custom loss functions defined!")

## 4. Load & Map Data

In [ ]:
def load_jsonl(filepath):
    """Load JSONL file into list of dictionaries."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Load the merged dataset
raw_data = load_jsonl(DATASET_PATH)
print(f"Total samples loaded: {len(raw_data)}")

df = pd.DataFrame(raw_data)

# Handle chat format (flatten as in v2)
if 'messages' in df.columns:
    print("Detected chat format, flattening...")
    df['text'] = df['messages'].apply(lambda x: x[1]['content'])  # User message
    df['intent'] = df['messages'].apply(lambda x: x[2]['content'])  # Assistant = intent
    df = df.drop(columns=['messages'])

# APPLY THE 5-CLASS MAPPING
print(f"\nApplying class reduction mapping...")
df['original_intent'] = df['intent'] # Keep original for reference
df['intent'] = df['intent'].map(lambda x: INTENT_MAPPING.get(x, "unknown"))

# Intent distribution
print(f"\nNew Intent Distribution:")
print("-" * 40)
counts = df['intent'].value_counts()
for intent, count in counts.items():
    pct = (count / len(df)) * 100
    bar = "█" * int(pct / 2)
    print(f"{intent:22s}: {count:4d} ({pct:5.1f}%) {bar}")

## 5. Data Augmentation

In [ ]:
def augment_minority_classes(df, min_samples=20):
    """
    Augment classes with fewer than min_samples examples.
    Uses simple text transformations suitable for Roman Urdu + English.
    """
    augmented_rows = []
    
    for intent in INTENT_LABELS:
        intent_df = df[df['intent'] == intent]
        count = len(intent_df)
        
        if count < min_samples:
            samples_needed = min_samples - count
            print(f"Augmenting '{intent}': {count} -> {min_samples} samples")
            
            for _ in range(samples_needed):
                # Randomly select a sample and augment it
                sample = intent_df.sample(1).iloc[0]
                text = sample['text']
                
                # Simple augmentation techniques for Roman Urdu + English
                augmented_text = text
                
                # 1. Add polite markers
                if random.random() < 0.3:
                    words = text.split()
                    if random.random() < 0.5:
                        augmented_text = " ".join(words + ["ji"])
                    else:
                        augmented_text = " ".join(["bhai"] + words)
                
                # 2. Case variation
                elif random.random() < 0.2:
                    augmented_text = text.lower()
                
                # 3. Add punctuation
                elif random.random() < 0.2:
                    if not text.endswith('?'):
                        augmented_text = text + "?"
                    elif not text.endswith('.'):
                        augmented_text = text + "."
                
                # 4. Synonym replacement
                elif random.random() < 0.2:
                    replacements = {
                        "book": "booking", "slot": "time slot", "hai": "he",
                        "kal": "tomorrow", "aaj": "today", "shaam": "evening",
                    }
                    for old, new in replacements.items():
                        if old in augmented_text.lower() and random.random() < 0.5:
                            augmented_text = augmented_text.replace(old, new)
                            break
                
                augmented_rows.append({
                    'text': augmented_text,
                    'intent': intent
                })
    
    if augmented_rows:
        augmented_df = pd.DataFrame(augmented_rows)
        df = pd.concat([df, augmented_df], ignore_index=True)
        print(f"\nTotal augmented samples added: {len(augmented_rows)}")
        print(f"New total: {len(df)} samples")
    else:
        print("No augmentation needed.")
    
    return df

# Apply augmentation
if CONFIG['augment_data']:
    print("\nApplying data augmentation...")
    print("=" * 50)
    df = augment_minority_classes(df, min_samples=CONFIG['min_samples_per_class'])
    
    print("\nUpdated Intent Distribution:")
    intent_counts_after = df['intent'].value_counts()
    for intent, count in intent_counts_after.items():
        print(f"  {intent:22s}: {count:4d}")
else:
    print("Data augmentation disabled.")

## 6. Data Preprocessing

In [ ]:
# Prepare data
texts = df['text'].tolist()
labels = [label2id[intent] for intent in df['intent'].tolist()]

print(f"Total samples: {len(texts)}")

# Stratified split
def stratified_split_with_min_samples(texts, labels, test_size=0.2, val_size=0.1, min_test_samples=2):
    # Split train vs temp
    train_texts, temp_texts, train_labels, temp_labels = train_test_split(
        texts, labels, test_size=0.2, random_state=SEED, stratify=labels
    )
    # Split validation vs test
    val_texts, test_texts, val_labels, test_labels = train_test_split(
        temp_texts, temp_labels, test_size=0.5, random_state=SEED, stratify=temp_labels
    )
    
    print(f"Train set: {len(train_texts)} samples")
    print(f"Validation set: {len(val_texts)} samples")
    print(f"Test set: {len(test_texts)} samples")
    
    return train_texts, val_texts, test_texts, train_labels, val_labels, test_labels

train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = stratified_split_with_min_samples(
    texts, labels, test_size=0.2, val_size=0.1
)

# Calculate class weights
train_labels_array = np.array(train_labels)
unique_labels = np.unique(train_labels_array)
class_weights = compute_class_weight('balanced', classes=unique_labels, y=train_labels_array)

class_weights_dict = {int(label): weight for label, weight in zip(unique_labels, class_weights)}
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.cuda()

print("\nClass Weights:")
for label_id, weight in class_weights_dict.items():
    print(f"{id2label[label_id]:22s}: {weight:.3f}")

In [ ]:
# Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Dataset creation
def create_hf_dataset(texts, labels):
    return HFDataset.from_dict({"text": texts, "label": labels})

dataset = DatasetDict({
    "train": create_hf_dataset(train_texts, train_labels),
    "validation": create_hf_dataset(val_texts, val_labels),
    "test": create_hf_dataset(test_texts, test_labels)
})

# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=CONFIG["max_length"]
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print("\nDataset tokenization successful.")

## 7. Model & Trainer Setup

In [ ]:
# Load Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(INTENT_LABELS),
    id2label=id2label,
    label2id=label2id
)

# Metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    _, _, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    
    return {"accuracy": accuracy, "macro_f1": macro_f1}

# Training Args
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"] * 2,
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=25,
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    seed=SEED,
)

# Custom Trainer
class CustomTrainer(Trainer):
    def __init__(self, loss_function=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_function = loss_function
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        if self.loss_function is not None:
            loss = self.loss_function(logits, labels)
        else:
            loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

# Init Trainer
loss_fn = FocalLoss(alpha=class_weights_tensor, gamma=CONFIG['focal_gamma']) if CONFIG['use_focal_loss'] else WeightedCrossEntropyLoss(class_weights_tensor)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    loss_function=loss_fn
)

print("Trainer ready.")

## 8. Training & Evaluation

In [ ]:
print("Starting training...")
train_result = trainer.train()
print(f"\nTraining time: {train_result.metrics['train_runtime']:.2f}s")
print(f"Best metric: {train_result.metrics.get('best_metric', 'N/A')}")

# Evaluate
print("\nEvaluating on Test Set...")
test_results = trainer.evaluate(tokenized_dataset["test"])
print(test_results)

# Detailed Report
predictions_output = trainer.predict(tokenized_dataset["test"])
predictions = np.argmax(predictions_output.predictions, axis=1)

print("\nClassification Report:")
print(classification_report(test_labels, predictions, target_names=INTENT_LABELS, zero_division=0))

## 9. Save Model

In [ ]:
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_SAVE_PATH))
tokenizer.save_pretrained(str(MODEL_SAVE_PATH))

# Save Config
label_config = {
    "labels": INTENT_LABELS,
    "label2id": label2id,
    "id2label": id2label,
    "model_name": MODEL_NAME,
    "config": CONFIG
}

with open(MODEL_SAVE_PATH / "label_config.json", "w") as f:
    json.dump(label_config, f, indent=2)

print(f"Model and config saved to: {MODEL_SAVE_PATH}")

## 10. Inference Test

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model=str(MODEL_SAVE_PATH), tokenizer=str(MODEL_SAVE_PATH), device=0 if torch.cuda.is_available() else -1)

test_samples = [
    "Aoa", 
    "book karna hai", 
    "charges kya hain?", 
    "yes confirm", 
    "random text"
]

print("Inference Check:")
for txt in test_samples:
    res = classifier(txt)[0]
    print(f"'{txt}' -> {res['label']} ({res['score']:.4f})")